# Aadhaar OCR Test

Extracts Aadhaar number, DOB, Gender, and Address from an Aadhaar card image using EasyOCR.

In [1]:
import cv2
import easyocr
import re
import os

reader = easyocr.Reader(['en', 'hi'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [2]:
def extract_aadhaar(image_path):
    if not os.path.exists(image_path):
        print(f"Error: Image file '{image_path}' not found!")
        return None

    print(f"Reading image: {image_path}")
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    print("Running text extraction...")
    results = reader.readtext(gray, detail=1)

    texts = [r[1] for r in results]
    full_text = " ".join(texts)

    print("\n--- Raw Text Extracted ---")
    for i, (bbox, text, conf) in enumerate(results):
        y = int(bbox[0][1])
        print(f"  [{i:>2}] [y={y:>4}] {text}")
    print("--------------------------\n")

    aadhaar_pattern = r'\b\d{4}\s?\d{4}\s?\d{4}\b'
    dob_pattern = r'(?:DOB|Date of Birth|Year of Birth)[:\s]*(\d{2}/\d{2}/\d{4}|\d{4})'
    gender_pattern = r'\b(Male|Female|MALE|FEMALE|TRANSGENDER)\b'
    name_pattern = r'Government of India[^A-Za-z]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'
    father_pattern = r'Father[:\s]*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)'

    aadhaar_matches = re.findall(aadhaar_pattern, full_text)
    aadhaar_number = aadhaar_matches[0].replace(' ', '') if aadhaar_matches else None

    # Card type: long cards have "Your Aadhaar No." label section
    aadhaar_label_idx = None
    for i, (bbox, text, _) in enumerate(results):
        if re.search(r'Aadhaar No', text):
            aadhaar_label_idx = i
            break

    card_type = "long" if aadhaar_label_idx is not None else "short"

    dob = re.search(dob_pattern, full_text, re.IGNORECASE)
    gender = re.search(gender_pattern, full_text, re.IGNORECASE)
    name = re.search(name_pattern, full_text)
    father = re.search(father_pattern, full_text, re.IGNORECASE)

    address = "Not Applicable"
    if aadhaar_label_idx is not None and aadhaar_label_idx > 4:
        addr_texts = []
        for i in range(4, aadhaar_label_idx):
            text = results[i][1].strip()
            if not text.isascii():
                continue
            if re.search(r'[A-Z]{2}\d{6,}', text):
                continue
            addr_texts.append(text)
        address = ", ".join(addr_texts) if addr_texts else "Not Found"

    result = {
        "status": "success",
        "card_type": card_type,
        "aadhaar_found": aadhaar_number is not None,
        "data": {
            "aadhaar_number": aadhaar_number,
            "name": name.group(1) if name else "Not Found",
            "father_name": father.group(1) if father else "Not Found",
            "dob": dob.group(1) if dob else "Not Found",
            "gender": gender.group(0).upper() if gender else "Not Found",
            "address": address
        }
    }

    print("Parsed JSON Result:")
    return result

In [3]:
image_path = "s.jpeg"
extract_aadhaar(image_path)

Reading image: s.jpeg
Running text extraction...


c:\Users\Anugrha Bhujel\miniconda3\envs\ocr\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Raw Text Extracted ---
  [ 0] [y= 146] To
  [ 1] [y= 172] तेजेन
  [ 2] [y= 174] बमो॰]
  [ 3] [y= 200] Telen Barman
  [ 4] [y= 192] है
  [ 5] [y= 228] PATHAR COLONY
  [ 6] [y= 259] GHOILAJOTE
  [ 7] [y= 262] MATIGARA
  [ 8] [y= 287] Gaurcharan
  [ 9] [y= 311] Matigara
  [10] [y= 343] Matigara Darjeeling
  [11] [y= 369] West Bengal 734010
  [12] [y= 348] ड्ूैँ
  [13] [y= 444] MA249552486F1
  [14] [y= 606] आपका
  [15] [y= 609] आधार
  [16] [y= 607] क्रमांक / Your Aadhaar No
  [17] [y= 660] 2152
  [18] [y= 665] 4563
  [19] [y= 667] 1287
  [20] [y= 740] आधार
  [21] [y= 749] आम
  [22] [y= 742] आदमी
  [23] [y= 751] का
  [24] [y= 741] अधिकार
  [25] [y= 851] भारत
  [26] [y= 854] सरकार
  [27] [y= 885] Government of India
  [28] [y= 924] तेजेन
  [29] [y= 924] बर्मन
  [30] [y= 956] Tejen Barman
  [31] [y= 982] पिता
  [32] [y= 984] भालो
  [33] [y= 982] बर्मन
  [34] [y=1014] Father
  [35] [y=1014] Bhalo Barman
  [36] [y=1047] ज॰्म
  [37] [y=1042] तिथि
  [38] [y=1044] DOB
  [39] [y=1041] 01/01/19

{'status': 'success',
 'card_type': 'long',
 'aadhaar_found': True,
 'data': {'aadhaar_number': '215245631287',
  'name': 'Tejen Barman',
  'father_name': 'Bhalo Barman',
  'dob': '01/01/1965',
  'gender': 'MALE',
  'address': 'PATHAR COLONY, GHOILAJOTE, MATIGARA, Gaurcharan, Matigara, Matigara Darjeeling, West Bengal 734010'}}